# Thesis plots

Loads on-disk seed-evaluation artifacts and produces figures.

Conventions used here:
- A *run* is one seed-eval **config directory** that contains `*_Seed{N}` subfolders. For tune mode that is `experiment_outputs/Tuning/<tune_name>/seed_evaluation/<config>/`; for single mode it is `experiment_outputs/Runs/<run>/seed_evaluation/`; for baselines it is `experiment_outputs/Runs/<baseline>/seed_evaluation/<baseline_kind>/`.
- All figures are constructed in plain matplotlib so that you can tweak each one in place.
- Mean / std curve aggregation reuses `compute_curve_stats_across_seeds` from `seed_evaluation.py` so the math matches what the in-run plotter saves to disk.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image

# Allow running the notebook from src/visualization/ by making the repo root importable.
REPO_ROOT = Path.cwd().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.experiments.utils.seed_evaluation import compute_curve_stats_across_seeds
from src.visualization.utils.plot_data_loader import load_run, load_episodes
from src.visualization.utils.eval_rollout_plots import generate_visualizations

## 1. Load runs

Add one entry per run you want to plot. The dict key becomes the legend label.

In [ ]:
run_paths = {
    # "PPO (tuned)":   REPO_ROOT / "experiment_outputs/Tuning/<tune>/seed_evaluation/01_xxx",
    # "PPO (default)": REPO_ROOT / "experiment_outputs/Runs/<run>/seed_evaluation",
    # "Baseline s,S":  REPO_ROOT / "experiment_outputs/Runs/<baseline>/seed_evaluation/sS",
}

runs = {label: load_run(path) for label, path in run_paths.items()}
for label, run in runs.items():
    print(f"{label}: {len(run['seed_data'])} seed(s) at {run['config_dir']}")

## 2. Training curves: mean ± std across seeds

One line per run. The shaded band is ±1 std across seeds at each iteration.

In [ ]:
metric = "eval_return"   # or "train_return"

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.tab10.colors

for idx, (label, run) in enumerate(runs.items()):
    seed_runs = [s["training_metrics"] for s in run["seed_data"].values() if s["training_metrics"]]
    if not seed_runs:
        continue
    iterations, means, stds = compute_curve_stats_across_seeds(seed_runs, key=metric)
    color = colors[idx % len(colors)]
    ax.plot(iterations, means, label=label, linewidth=1.5, color=color)
    ax.fill_between(iterations, means - stds, means + stds, alpha=0.2, color=color)

ax.set_xlabel("Training Iteration")
ax.set_ylabel(f"{metric} (mean ± std)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

## 3. Per-seed individual lines for a single run

Useful for seed-stability diagnostics.

In [ ]:
label = next(iter(runs))    # change to a specific run label
run = runs[label]
metric = "eval_return"

fig, ax = plt.subplots(figsize=(10, 5))
for seed_number, seed in sorted(run["seed_data"].items()):
    metrics = seed["training_metrics"]
    if not metrics:
        continue
    xs = [m["iteration"] for m in metrics]
    ys = [m.get(metric) for m in metrics]
    ax.plot(xs, ys, alpha=0.7, linewidth=1.0, label=f"seed={seed_number}")

ax.set_xlabel("Training Iteration")
ax.set_ylabel(metric)
ax.set_title(f"{label} — per-seed {metric}")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)
plt.tight_layout()

## 4. Final eval return per run (bar chart with std error bars)

Reads from each run's `seed_evaluation_summary.yaml` if present, else aggregates from `eval_results_final.yaml` files.

In [ ]:
labels, means, stds = [], [], []
for label, run in runs.items():
    summary = run["summary"]
    if summary and summary.get("configs"):
        cfg = summary["configs"][0]
        means.append(cfg["mean"]); stds.append(cfg["std"])
    else:
        values = [s["final_eval"]["episode_return_mean"]
                  for s in run["seed_data"].values()
                  if s["final_eval"] and s["final_eval"].get("episode_return_mean") is not None]
        if not values:
            continue
        means.append(float(np.mean(values)))
        stds.append(float(np.std(values, ddof=1)) if len(values) > 1 else 0.0)
    labels.append(label)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(labels))
ax.bar(x, means, yerr=stds, capsize=4, color=plt.cm.tab10.colors[:len(labels)], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylabel("Final Eval Return (mean ± std)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()

## 5. Baselines as horizontal bands on a curve plot

Useful for overlaying non-training baselines on top of training-curve figures.

In [ ]:
baseline_labels = []   # populate with labels in `runs` that should be drawn as bands

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.tab10.colors

# Curves for non-baseline runs
for idx, (label, run) in enumerate(runs.items()):
    if label in baseline_labels:
        continue
    seed_runs = [s["training_metrics"] for s in run["seed_data"].values() if s["training_metrics"]]
    if not seed_runs:
        continue
    iterations, means, stds = compute_curve_stats_across_seeds(seed_runs, key="eval_return")
    color = colors[idx % len(colors)]
    ax.plot(iterations, means, label=label, linewidth=1.5, color=color)
    ax.fill_between(iterations, means - stds, means + stds, alpha=0.2, color=color)

# Horizontal band for each baseline
for jdx, label in enumerate(baseline_labels):
    summary = runs[label]["summary"]
    if not (summary and summary.get("configs")):
        continue
    cfg = summary["configs"][0]
    color = colors[(len(runs) + jdx) % len(colors)]
    ax.axhline(cfg["mean"], linestyle="--", color=color, label=label, linewidth=1.5)
    ax.axhspan(cfg["mean"] - cfg["std"], cfg["mean"] + cfg["std"], alpha=0.15, color=color)

ax.set_xlabel("Training Iteration")
ax.set_ylabel("Eval Return (mean ± std)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

## 6. Per-policy rollout visualization (optional)

Two ways:

- **Existing PNG** — display the per-seed visualization that was already saved during evaluation.
- **Re-plot from `episodes_data.npz`** — reload the raw rollout and call `generate_visualizations` with a fresh output directory. Requires that the run was evaluated with `--visualize`.

In [ ]:
# Option A: display an existing per-seed PNG.
label = next(iter(runs))
seed_number = next(iter(runs[label]["seed_data"]))
seed_dir = runs[label]["seed_data"][seed_number]["seed_run_dir"]
viz_dirs = sorted((seed_dir / "visualizations").glob("visualization_*"))
if viz_dirs:
    Image(filename=str(viz_dirs[-1] / "01_inventory.png"))

In [ ]:
# Option B: reload episodes_data.npz and re-run the full visualization pipeline
# into a temporary output directory you control.
import tempfile

label = next(iter(runs))
seed_number = next(iter(runs[label]["seed_data"]))
seed_dir = runs[label]["seed_data"][seed_number]["seed_run_dir"]

episodes = load_episodes(seed_dir)
with tempfile.TemporaryDirectory() as tmp:
    generate_visualizations(episodes, tmp)
    Image(filename=str(Path(tmp) / "01_inventory.png"))